# 10 - Stage 11, part B: the report sections, the traceability and the old against new results

**Purpose.** The sections of the submitted report that are text (S-01, S-05, S-42 to S-44) rewritten from the
frozen final run `20261114T083930Z_final` (tag `g1-frozen`), the table that lists every number of the text with
its source, and the table of old against new results.

**Rules.**
- Nothing is trained and the step `test` is not run again.
- Every test number is placed beside the validation number of the same view.
- No winner is declared (D-37). Four of the five neural models have one seed.
- Decisions of the user (2026-11-14): Q-29, the dust-day half of L-20 is dropped; Q-30, a new script rebuilds
  the tables from the frozen predictions and `reproduce_paper_results.py` is not edited.

**Documented in** `docs/stage11_plan.md` section 5, OP-67, run journal R11-02, errata C-85 to C-91 (M51 to M57)
and X107 to X112, tests T-178 to T-184, and the figures S11-18, S11-19, C-11 and C-12.

## 1. What the generator wrote

In [1]:
import pandas as pd

from solarpv.config.paths import REPO_ROOT

DOCS = REPO_ROOT / "docs"
tr = pd.read_csv(DOCS / "traceability.csv", dtype=str, keep_default_na=False)
report = (DOCS / "report_v7.md").read_text(encoding="utf-8")
files = tr["file"].str.split("; ").str[0]
pd.Series({"registered numbers": len(tr), "source files": files.nunique(),
           "sections": sum(ln.startswith("## ") for ln in report.splitlines()),
           "lines of text": len(report.splitlines())})

registered numbers    384
source files           79
sections               30
lines of text         862
dtype: int64

**What it shows.** `scripts/make_report_v7.py` wrote 30 sections (862
lines) and 384 registered numbers, from 79 files.

**What it means.** A number cannot reach the text without a row: each is formatted by the generator with its file, its
script and the date, and the test T-178 fails when the text holds a number that no row names. 43 rows are
quotations of the submitted report or of the project documents, registered as old values; all other numbers are read
from a file of the run or of `outputs/` (S11-18).

## 2. The route, old against new (C2, M52)

In [2]:
ov = pd.read_csv(DOCS / "old_vs_new.csv", dtype=str, keep_default_na=False)
r = ov[ov["section"] == "route"][["item", "old_value", "new_validation", "new_test"]]
r.set_index("item")

,old_value,new_validation,new_test
item,,,
"XGB-Opt D+1, direct minus indirect",+1.98,-0.00,-0.04
"XGB-Opt D+2, direct minus indirect",+2.01,-0.07,+0.05
"XGB-Opt D+3, direct minus indirect",+2.10,-0.12,+0.11
"BiLSTM-Opt D+1, direct minus indirect",+1.66,-0.36,-0.19
"BiLSTM-Opt D+2, direct minus indirect",+1.72,-0.37,-0.35
"BiLSTM-Opt D+3, direct minus indirect",+1.89,-0.37,-0.31


**What it shows.** The nRMSE of the direct route minus that of the indirect route in kW (a positive value:
the indirect route is better) for XGB-Opt and BiLSTM-Opt, as submitted and in the rebuild on validation and on test.

**What it means.** The submitted report found the indirect route better by 1.66 to
2.10 pp. In the rebuild the difference is -0.37 to
-0.00 pp on validation and -0.35 to
+0.11 pp on test, and the sign changes between the periods for 3 of the 6 pairs. The
submitted truth for the indirect route was rebuilt from the clipped index (C-16); the advantage is not reproduced and
neither route is declared better (`C-12`).

## 3. The benchmark, old against new

In [3]:
b = ov[ov["section"] == "benchmark"]
cols = ["item", "old_value", "new_validation", "new_test", "comparable"]
b[b["item"].str.contains("indirect D\\+1")][cols].set_index("item")

,old_value,new_validation,new_test,comparable
item,,,,
XGB-Opt indirect D+1,29.09 (n=2580),13.72,21.70,"units differ (k is rescaled and unclipped, F-06)"
BiLSTM-Opt indirect D+1,30.24 (n=2510),13.92,22.50,"units differ (k is rescaled and unclipped, F-06)"
TFT-96 indirect D+1,29.66 (n=2554),13.25,21.79,"units differ (k is rescaled and unclipped, F-06)"
TFT-168 indirect D+1,31.05 (n=2554),13.29,21.42,"units differ (k is rescaled and unclipped, F-06)"
TFT baseline indirect D+1,31.76 (n=2554),13.40,21.67,"units differ (k is rescaled and unclipped, F-06)"
PatchTST indirect D+1,30.30 (n=2580),17.30,26.65,"units differ (k is rescaled and unclipped, F-06)"


**What it shows.** The D+1 indirect nRMSE of the six models: the submitted value (in the units of the target, with the
number of hours), and the rebuild's value in the same unit on validation and on test (seed mean).

**What it means.** The table has 36 rows in the units of the target, and a second set in kW. The
indirect rows are marked as not comparable: the target changed (F-06: the index is not clipped and its mean is
0.788 instead of 0.6279), so the submitted 29.09 per cent and the rebuilt value are not a difference of one quantity.
27 of the 92 rows of the whole table carry such a mark, and the reason is in the column next to it
(X108).

## 4. Where the numbers come from (S11-18)

In [4]:
first = tr["file"].str.split("; ").str[0]
fam = first.map(lambda f: "quoted" if ("v6.docx" in f or "PROJECT_MASTER" in f)
                else "analyses" if "11_analyses" in f else "run or stages")
fam.value_counts()

file
run or stages    292
analyses          49
quoted            43
Name: count, dtype: int64

**What it shows.** The registered numbers by kind of source file.

**What it means.** 43 of the 384 numbers are quotations of fixed documents (the submitted report and the
project documents) and the rest come from output files. The figure `S11-18` draws the same counts.

## 5. The statements the sections make about the results (D-37)

In [5]:
t = report
checks = {
    "no winner declared": "No model is declared the winner" in t,
    "validation beside test": "beside the validation number of the same view" in t,
    "operation-limited hours named": "operation-limited" in t,
    "quintile profile is a binning effect": "made by the binning" in t,
    "climatology margin stated": "minimum detectable effect" in t,
}
pd.Series(checks)

no winner declared                      True
validation beside test                  True
operation-limited hours named           True
quintile profile is a binning effect    True
climatology margin stated               True
dtype: bool

**What it shows.** Five phrases of the rules for the results, searched in the generated text.

**What it means.** All 5 are present: no
winner is declared (D-37), every test number has the validation number of the same view beside it, the positive bias is
placed in the operation-limited hours (F-30), the quintile profile of v6 is called an effect of the binning and not
physics (M45), and the limit of the test against climatology is stated through the minimum detectable effect. The
test T-181 fails if one of them is removed.

## 6. What part B did not do

- **Dust days (L-20).** Dropped by the user (Q-29, option c): the project holds no aerosol record, and a label made from
  the measured deficit would be made from the outcome. The season half of L-20 stays.
- **`--xgb-only` (S-45).** A training run, left for the end of stage 11 (Q-30, option a). The rebuild of the tables and
  figures without training is `scripts/make_tables_from_frozen.py`.
- **The radar of v6 figure 14** (C-84) and the **coverage of the quantile intervals** (L-14) are not part of the scope
  of this part.